# NVT+W / TMMC simulation grid workflow

This notebook contains three related workflows:

1. **Create simulations:** generate a grid of simulation folders from a `base/` directory and set the particle numbers for each component.
2. **Analyze simulations:** read `tmmc/tmmc_statistics.txt`, construct the transition matrix, reconstruct the particle-number distribution, and inspect convergence.
3. **Reweight and build isotherms:** reweight the reconstructed distribution from a reference fugacity to target fugacities and compute mixture loadings.

The method assumes a directory layout like this:

```text
project/
├── base/
│   ├── simulation.json
│   ├── <component definition files>
│   ├── <framework CIF>
│   └── <force-field JSON files>
├── 0_0/
├── 0_4/
├── 4_0/
└── ...
```

Set up `base/` before running the creation step. It should contain a complete default simulation: component definitions, framework CIF, force-field JSON files, and a `simulation.json` file.


## Theoretical background

The workflow uses a grand-canonical description of adsorption, but the simulations are run on a fixed particle-number grid. Each grid point stores transition-matrix statistics for neighboring macrostates. Those local transition probabilities are then stitched together to reconstruct the grand-canonical particle-number distribution.

Let the component particle-number vector be

$$
\mathbf{N}=(N_1,N_2,\ldots,N_{n_c}),
$$

where \(n_c\) is the number of adsorbing components. The inverse thermal energy is

$$
\beta = \frac{1}{k_B T}.
$$


### Grand-canonical macrostate probability

For one component at fugacity \(f\), volume \(V\), and temperature \(T\), the grand-canonical partition function can be written as

$$
\Xi(f,V,T)
=
\sum_{N=0}^{\infty}
\frac{(\beta f V)^N}{N!}
\int \exp[-\beta H(q;N)]\,dq.
$$

The particle-number probability is the contribution from macrostate \(N\), normalized by \(\Xi\):

$$
\Pi(N;f,V,T)
=
\frac{1}{\Xi(f,V,T)}
\frac{(\beta f V)^N}{N!}
\int \exp[-\beta H(q;N)]\,dq.
$$

For a binary mixture, the corresponding probability is

$$
\Pi(N_1,N_2;f_1,f_2,V,T)
=
\frac{1}{\Xi}
\frac{(\beta f_1 V)^{N_1}}{N_1!}
\frac{(\beta f_2 V)^{N_2}}{N_2!}
\int \exp[-\beta H(q;N_1,N_2)]\,dq.
$$

More generally,

$$
\Pi(\mathbf{N};\mathbf{f},V,T)
=
\frac{1}{\Xi}
\left[\prod_{i=1}^{n_c}\frac{(\beta f_i V)^{N_i}}{N_i!}\right]
\int \exp[-\beta H(q;\mathbf{N})] \, dq.
$$

Only the factor \(\prod_i f_i^{N_i}\) changes when fugacity is changed at fixed \(V\), \(T\), force field, and framework. This is the basis for fugacity reweighting later in the notebook.


### Transition-matrix reconstruction

For a two-component particle-number grid, each macrostate is \(\mathbf{N}=(N_1,N_2)\). The neighboring moves are

$$
\boldsymbol{\delta} \in \{(-1,0),(+1,0),(0,-1),(0,+1)\}.
$$

The notebook stores the transition-count vector as

$$
C(\mathbf{N})=
\left[
C^0(\mathbf{N}),
C_1^-(\mathbf{N}),
C_1^+(\mathbf{N}),
C_2^-(\mathbf{N}),
C_2^+(\mathbf{N})
\right],
$$

where \(C_i^+(\mathbf{N})\) is the accumulated acceptance count for \(\mathbf{N}\rightarrow\mathbf{N}+\mathbf{e}_i\), \(C_i^-(\mathbf{N})\) is the count for \(\mathbf{N}\rightarrow\mathbf{N}-\mathbf{e}_i\), and \(C^0(\mathbf{N})\) is the stay/rejection count. In the code, this is read from

```python
cmatrix = tmmc_results[..., 3:8]
```

The normalized transition probability for a neighboring macrostate is

$$
p(\mathbf{N}\rightarrow\mathbf{N}+\boldsymbol{\delta})
=
\frac{C^{\boldsymbol{\delta}}(\mathbf{N})}
{\sum_{\boldsymbol{\delta}'} C^{\boldsymbol{\delta}'}(\mathbf{N})}.
$$

Detailed balance connects neighboring macrostates:

$$
\Pi(\mathbf{N})\,
p(\mathbf{N}\rightarrow\mathbf{N}+\boldsymbol{\delta})
=
\Pi(\mathbf{N}+\boldsymbol{\delta})\,
p(\mathbf{N}+\boldsymbol{\delta}\rightarrow\mathbf{N}).
$$

Therefore, the log-probability update is

$$
\ln \Pi(\mathbf{N}+\boldsymbol{\delta})
=
\ln \Pi(\mathbf{N})
+
\ln\left[
\frac{p(\mathbf{N}\rightarrow\mathbf{N}+\boldsymbol{\delta})}
{p(\mathbf{N}+\boldsymbol{\delta}\rightarrow\mathbf{N})}
\right].
$$

A common weighted-residual form for combining all forward/reverse links is:

$$
\Delta^2
=
\sum_{\mathbf{N}}
\sum_{\boldsymbol{\delta}}
\sqrt{
C^{\boldsymbol{\delta}}(\mathbf{N})
C^{-\boldsymbol{\delta}}(\mathbf{N}+\boldsymbol{\delta})
}
\left[
\ln\left(
\frac{p(\mathbf{N}\rightarrow\mathbf{N}+\boldsymbol{\delta})}
{p(\mathbf{N}+\boldsymbol{\delta}\rightarrow\mathbf{N})}
\right)
-
\ln\left(
\frac{\Pi(\mathbf{N}+\boldsymbol{\delta})}{\Pi(\mathbf{N})}
\right)
\right]^2.
$$

This implementation uses a direct graph propagation of the same nearest-neighbor detailed-balance ratios. States that are disconnected from the first valid macrostate remain zero-probability states.


### Log normalization and numerical stability

The reconstructed values are handled as log weights. If \(a(\mathbf{N})\) is an unnormalized log probability, the normalized distribution is

$$
\Pi(\mathbf{N})
=
\frac{\exp[a(\mathbf{N})]}
{\sum_{\mathbf{M}}\exp[a(\mathbf{M})]}.
$$

For numerical stability, the implementation subtracts the largest finite log weight before exponentiation:

$$
\ln \Pi(\mathbf{N})
=
a(\mathbf{N})
-
\left[
\max_{\mathbf{M}} a(\mathbf{M})
+
\ln\sum_{\mathbf{M}}\exp\left(a(\mathbf{M})-\max_{\mathbf{K}}a(\mathbf{K})\right)
\right].
$$


### Fugacity reweighting and adsorption isotherms

Once \(\Pi(\mathbf{N};\mathbf{f},V,T)\) is known at a reference fugacity vector \(\mathbf{f}\), a target fugacity vector \(\mathbf{f}'\) is obtained from

$$
\ln \tilde{\Pi}(\mathbf{N};\mathbf{f}',V,T)
=
\ln \Pi(\mathbf{N};\mathbf{f},V,T)
+
\sum_{i=1}^{n_c} N_i\ln\left(\frac{f_i'}{f_i}\right).
$$

The tilde indicates that this is not yet normalized. The normalized reweighted distribution is

$$
\Pi(\mathbf{N};\mathbf{f}',V,T)
=
\frac{\exp[\ln \tilde{\Pi}(\mathbf{N};\mathbf{f}',V,T)]}
{\sum_{\mathbf{M}}\exp[\ln \tilde{\Pi}(\mathbf{M};\mathbf{f}',V,T)]}.
$$

The component loading is the expectation value over the reweighted distribution:

$$
\langle N_i\rangle_{\mathbf{f}'}
=
\sum_{\mathbf{N}} N_i\,\Pi(\mathbf{N};\mathbf{f}',V,T).
$$

An adsorption isotherm is obtained by evaluating this expectation over a series of target fugacities,

$$
\mathbf{f}'^{(0)},\mathbf{f}'^{(1)},\ldots,\mathbf{f}'^{(n)},
$$

and plotting \(\langle N_i\rangle\) versus fugacity for each component.



### Practical checks

The particle-number grid must be large enough that high-fugacity distributions do not pile up at the maximum sampled particle number. If the probability maximum reaches \(N_i^{\max}\), the isotherm can show an artificial plateau or a clipped peak. A preliminary high-fugacity GCMC calculation is a useful way to estimate a safe \(N_i^{\max}\) before launching the full TMMC/NVT+W grid.

For mixtures, the total number of particles can become computationally expensive even when each individual component remains below its own maximum. The creation and analysis functions therefore support a `max_total_particles` cutoff. Grid points with

$$
\sum_i N_i > N_\mathrm{total}^{\max}
$$

are skipped during simulation-folder creation and are not counted as missing during result ingestion.

Smooth \(\ln\Pi\) curves and smooth free-energy slices are useful diagnostics. Sharp jumps typically indicate missing simulations, disconnected macrostates, or very small forward/reverse transition counts.


## Example `base/simulation.json`

The creation function starts from this type of default Monte Carlo input and converts it into one fixed-composition TMMC/NVT+W case per particle-number grid point.

```json
{
    "SimulationType": "MonteCarlo",
    "NumberOfCycles": 100000,
    "NumberOfInitializationCycles": 10000,
    "NumberOfEquilibrationCycles": 10000,
    "PrintEvery": 1000,
    "Systems": [
        {
            "Type": "Framework",
            "Name": "CALF-20",
            "NumberOfUnitCells": [3, 3, 3],
            "ExternalTemperature": 297.0,
            "ExternalPressure": 97000,
            "UseChargesFrom": "CIF_File",
            "ChargeMethod": "Ewald"
        }
    ],
    "Components": [
        {
            "Name": "h2o",
            "FugacityCoefficient": 1.0,
            "TranslationProbability": 1.0,
            "RotationProbability": 1.0,
            "ReinsertionProbability": 1.0,
            "SwapProbability": 1.0,
            "CreateNumberOfMolecules": 0
        },
        {
            "Name": "co2",
            "FugacityCoefficient": 1.0,
            "TranslationProbability": 1.0,
            "RotationProbability": 1.0,
            "ReinsertionProbability": 1.0,
            "SwapProbability": 1.0,
            "CreateNumberOfMolecules": 0
        }
    ]
}
```


In [ ]:
import json
import os
import shutil
from collections import deque
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


## User settings

`minmax_particles` defines the sampled particle-number range for each component. The notebook uses `np.arange(nmin, nmax + 1, deltaN)`, so the upper bound is included.

`max_total_particles` limits the total number of particles in a generated case. For a binary mixture, a grid point is included only if

$$
N_1 + N_2 \leq N_\mathrm{total}^{\max}.
$$

Set `max_total_particles = None` to disable this total-particle cutoff.

With the default values below, the nominal rectangular grid is:

- H₂O: `0, 4, 8, ..., 100`
- CO₂: `0, 4, 8, ..., 100`

The active simulation grid is the subset of those points that also satisfies `N_H2O + N_CO2 <= max_total_particles`.

`component_ids = [0, 1]` means that the first entry in `minmax_particles` is applied to `Components[0]`, and the second entry is applied to `Components[1]` in `simulation.json`.


In [ ]:
minmax_particles = [[0, 100], [0, 100]]
component_ids = [0, 1]
deltaN = 4
max_total_particles = None

base_dir = "base"
output_dir = "."

component_names = ["H2O", "CO2"]
component_labels = [r"H$_2$O", r"CO$_2$"]

limits = [np.arange(nmin, nmax + 1, deltaN) for nmin, nmax in minmax_particles]
grid_shape = [len(limit) for limit in limits]
grid_point_count = int(np.prod(grid_shape))

if max_total_particles is None:
    active_grid_point_count = grid_point_count
else:
    active_grid_point_count = sum(
        sum(particles) <= max_total_particles for particles in product(*limits)
    )

component_settings = {
    "TranslationProbability": 1.0,
    "RotationProbability": 1.0,
    "ReinsertionProbability": 1.0,
    "WidomProbability": 1.0,
    "SwapProbability": 1.0,
}

simulation_settings = {
    "SimulationType": "MonteCarloTransitionMatrix",
    "NumberOfCycles": 10000,
    "NumberOfInitializationCycles": 10000,
    "NumberOfEquilibrationCycles": 10000,
    "PrintEvery": 1000,
}

print(grid_shape, "grid points per component")
print(grid_point_count, "nominal grid points")
print(active_grid_point_count, "simulations after max_total_particles cutoff")
print(grid_point_count - active_grid_point_count, "grid points skipped by cutoff")


## Create simulations

The `create` function copies every file and subdirectory from `base/` into each simulation folder, then edits the copied `simulation.json`.

For each accepted particle tuple, for example `(40, 12)`, the function creates a directory named `40_12`. If `max_total_particles` is set, tuples with `sum(particles) > max_total_particles` are skipped and no folder is created. Inside each accepted directory, the function sets:

- `SimulationType = "MonteCarloTransitionMatrix"`
- `CreateNumberOfMolecules = n`
- `MinMacrostate = n`
- `MaxMacrostate = n`
- move probabilities from `component_settings`

`MinMacrostate` and `MaxMacrostate` are both set to the same particle number. This creates a fixed-composition NVT-like state at each grid point while still collecting the TMMC/Widom statistics needed to reconstruct the particle-number distribution.

Run the dry run first. It checks the grid, applies the total-particle cutoff, and reports the folders that would be created without writing files.


In [ ]:
def set_case_checked(settings_dict, key, value):
    """
    Set a JSON key while removing duplicate keys that differ only by case.

    This prevents inputs such as `SimulationType` and `simulationtype` from
    co-existing after programmatic edits.
    """
    duplicate_keys = [
        existing_key
        for existing_key in settings_dict
        if existing_key.lower() == key.lower() and existing_key != key
    ]

    for existing_key in duplicate_keys:
        del settings_dict[existing_key]

    settings_dict[key] = value


def particle_tuple_allowed(particles, max_total_particles=None):
    """Return True when a particle tuple satisfies the total-particle cutoff."""
    if max_total_particles is None:
        return True

    return sum(particles) <= max_total_particles


def copy_base_directory(base_path, sim_path, overwrite=True):
    """
    Copy the contents of `base_path` into `sim_path`.

    Existing simulation output directories such as `tmmc/` are preserved. Files
    from `base_path` overwrite existing files when `overwrite=True`.
    """
    for name in os.listdir(base_path):
        src = base_path / name
        dst = sim_path / name

        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True)
        elif overwrite or not dst.exists():
            shutil.copy2(src, dst)


def validate_create_inputs(base_path, minmax_particles, component_ids, max_total_particles=None):
    """Validate settings before creating a grid of simulations."""
    if not base_path.exists():
        raise FileNotFoundError(f"base directory not found: {base_path}")

    simulation_json = base_path / "simulation.json"
    if not simulation_json.exists():
        raise FileNotFoundError(f"missing required file: {simulation_json}")

    if len(minmax_particles) != len(component_ids):
        raise ValueError("minmax_particles and component_ids must have the same length")

    if max_total_particles is not None and max_total_particles < 0:
        raise ValueError("max_total_particles must be non-negative or None")

    with open(simulation_json) as f:
        settings = json.load(f)

    if "Components" not in settings:
        raise KeyError("base simulation.json must contain a Components list")

    n_components = len(settings["Components"])
    for component_id in component_ids:
        if component_id < 0 or component_id >= n_components:
            raise IndexError(
                f"component_id {component_id} is out of range for {n_components} components"
            )


def create(
    minmax_particles,
    component_ids,
    deltaN=4,
    base_dir="base",
    output_dir=".",
    component_settings=None,
    simulation_settings=None,
    max_total_particles=None,
    overwrite=True,
    dry_run=True,
):
    """
    Create a grid of fixed-composition TMMC/NVT+W simulation directories.

    Parameters
    ----------
    minmax_particles : list[list[int, int]]
        Per-component particle ranges as `[nmin, nmax]`. The upper bound is included.
    component_ids : list[int]
        Indices into `settings["Components"]` that receive the corresponding particle count.
    deltaN : int, default 4
        Spacing between particle counts in each dimension.
    base_dir : str or pathlib.Path, default "base"
        Directory containing the default simulation files.
    output_dir : str or pathlib.Path, default "."
        Directory where the `n_m` simulation folders are created.
    component_settings : dict or None
        Component-level settings to apply to every selected component.
    simulation_settings : dict or None
        Top-level simulation settings to apply to every generated `simulation.json`.
    max_total_particles : int, float, or None, default None
        If set, skip particle tuples with `sum(particles) > max_total_particles`.
    overwrite : bool, default True
        Whether copied base files overwrite existing files in simulation folders.
    dry_run : bool, default True
        If True, return planned folder paths without creating or editing files.

    Returns
    -------
    created_dirs : list[pathlib.Path]
        Paths for the generated or planned simulation directories that satisfy the cutoff.
    """
    base_path = Path(base_dir)
    output_path = Path(output_dir)

    component_settings = dict(component_settings or {})
    simulation_settings = dict(simulation_settings or {})

    validate_create_inputs(
        base_path=base_path,
        minmax_particles=minmax_particles,
        component_ids=component_ids,
        max_total_particles=max_total_particles,
    )

    limits = [np.arange(nmin, nmax + 1, deltaN) for nmin, nmax in minmax_particles]
    created_dirs = []
    skipped_dirs = []

    for particles in product(*limits):
        particles = [int(n) for n in particles]
        simdir = "_".join(str(n) for n in particles)
        sim_path = output_path / simdir

        if not particle_tuple_allowed(particles, max_total_particles=max_total_particles):
            skipped_dirs.append(sim_path)
            continue

        created_dirs.append(sim_path)

        if dry_run:
            continue

        sim_path.mkdir(parents=True, exist_ok=True)
        copy_base_directory(base_path, sim_path, overwrite=overwrite)

        simulation_json = sim_path / "simulation.json"
        with open(simulation_json) as f:
            settings = json.load(f)

        for key, value in simulation_settings.items():
            set_case_checked(settings, key, value)

        for component_id, n in zip(component_ids, particles):
            component = settings["Components"][component_id]

            set_case_checked(component, "CreateNumberOfMolecules", n)
            set_case_checked(component, "MinMacrostate", n)
            set_case_checked(component, "MaxMacrostate", n)

            for key, value in component_settings.items():
                set_case_checked(component, key, value)

        with open(simulation_json, "w") as f:
            json.dump(settings, f, indent=4)
            f.write("\n")

    action = "Planned" if dry_run else "Created/updated"
    print(f"{action} {len(created_dirs)} simulation directories")

    if max_total_particles is not None:
        print(
            f"Skipped {len(skipped_dirs)} grid points with total particles "
            f"> {max_total_particles}"
        )

    return created_dirs


In [ ]:
# Dry run: verify the folder grid before writing anything.
created_dirs = create(
    minmax_particles=minmax_particles,
    component_ids=component_ids,
    deltaN=deltaN,
    base_dir=base_dir,
    output_dir=output_dir,
    component_settings=component_settings,
    simulation_settings=simulation_settings,
    max_total_particles=max_total_particles,
    dry_run=True,
)

created_dirs[:5], created_dirs[-5:]


In [ ]:
# After checking the dry-run output, set dry_run=False to create or update folders.
# created_dirs = create(
#     minmax_particles=minmax_particles,
#     component_ids=component_ids,
#     deltaN=deltaN,
#     base_dir=base_dir,
#     output_dir=output_dir,
#     component_settings=component_settings,
#     simulation_settings=simulation_settings,
#     max_total_particles=max_total_particles,
#     dry_run=False,
# )


## Running the simulations

After the folders are created, run the simulation engine from each generated directory using your normal execution method.

Each completed case is expected to write:

```text
<n_h2o>_<n_co2>/tmmc/tmmc_statistics.txt
```

The analysis cells below skip missing files, so the notebook can be run while a grid is only partially complete. Missing accepted cases remain `NaN` in `tmmc_results`, and disconnected or unobserved states receive zero probability in `pi`. Grid points excluded by `max_total_particles` are skipped during result ingestion and are not counted as missing simulations.


## Analyze TMMC statistics

The theoretical background above defines the macrostate probability, transition-count normalization, detailed-balance reconstruction, and reweighting equations. This analysis block implements those equations for the dense grid stored in `tmmc_results`.

The analysis assumes that columns `3:8` of `tmmc_statistics.txt` contain the five transition-count entries used for a two-component grid:

```python
cmatrix = tmmc_results[..., 3:8]
```

The column convention is:

- `cmatrix[..., 0]`: rejection/stay count, \(C^0\)
- `cmatrix[..., 1]`: transition count toward lower component-0 macrostate, \(C_1^-\)
- `cmatrix[..., 2]`: transition count toward higher component-0 macrostate, \(C_1^+\)
- `cmatrix[..., 3]`: transition count toward lower component-1 macrostate, \(C_2^-\)
- `cmatrix[..., 4]`: transition count toward higher component-1 macrostate, \(C_2^+\)

`optimize_pi` reconstructs \(\ln\Pi\), not \(\Pi\), to avoid underflow on large grids. Convert it to probabilities only after log normalization:

$$
\Pi(\mathbf{N})=\exp[\ln\Pi(\mathbf{N})].
$$

Accepted grid points whose result file is not found are reported in `missing`. Grid points excluded by `max_total_particles` are reported in `skipped` and are not treated as failed or missing simulations.


In [ ]:
def normalize_cmatrix(cmatrix):
    """Normalize transition counts into transition probabilities, preserving missing states."""
    norm = np.nansum(cmatrix, axis=-1, keepdims=True)
    pmatrix = np.zeros_like(cmatrix, dtype=float)

    valid = np.squeeze(norm, axis=-1) > 0.0
    pmatrix[valid] = np.nan_to_num(cmatrix[valid], nan=0.0) / norm[valid]

    return pmatrix


def log_normalize(ln_values):
    """
    Normalize log weights.

    Entries equal to `-np.inf` remain zero-probability states after exponentiation.
    """
    ln_values = np.asarray(ln_values, dtype=float).copy()
    finite = np.isfinite(ln_values)

    if not np.any(finite):
        raise ValueError("cannot normalize: all log weights are non-finite")

    max_ln_value = np.max(ln_values[finite])
    log_partition = max_ln_value + np.log(np.sum(np.exp(ln_values[finite] - max_ln_value)))

    ln_values[finite] -= log_partition
    return ln_values


def macrostate_grids(limits):
    """Return one broadcast particle-number grid per component."""
    return np.meshgrid(*[np.asarray(limit, dtype=float) for limit in limits], indexing="ij")


def as_fugacity_array(fugacity, dims, name):
    """
    Convert a scalar or per-component fugacity input into a length-`dims` array.

    A scalar fugacity is applied to every component. A vector fugacity must have
    one positive value per component.
    """
    fugacity = np.asarray(fugacity, dtype=float).reshape(-1)

    if fugacity.size == 1:
        fugacity = np.repeat(fugacity, dims)
    elif fugacity.size != dims:
        raise ValueError(f"{name} must be a scalar or have length {dims}")

    if np.any(fugacity <= 0.0):
        raise ValueError(f"{name} must contain strictly positive fugacities")

    return fugacity


def optimize_pi(C):
    """
    Estimate the normalized log particle-number distribution from a TMMC count matrix.

    The estimator propagates log probabilities through the macrostate grid using
    nearest-neighbor detailed-balance ratios. States with no usable path from the
    first valid state remain at `-np.inf`, i.e. zero probability after exponentiation.
    """
    sizes = C.shape[:-1]
    dims = len(sizes)
    n_transitions = 1 + 2 * dims

    if C.shape[-1] < n_transitions:
        raise ValueError(
            f"C has {C.shape[-1]} transition columns; expected at least {n_transitions}"
        )

    C = np.nan_to_num(C[..., :n_transitions], nan=0.0, posinf=0.0, neginf=0.0)

    norm = C.sum(axis=-1)
    mask = norm > 1e-30

    if not np.any(mask):
        raise ValueError("transition matrix contains no non-zero count rows")

    p = np.zeros_like(C, dtype=float)
    p[mask] = C[mask] / norm[mask, None]

    lnpi = np.full(sizes, -np.inf, dtype=float)
    computed = np.zeros(sizes, dtype=bool)

    anchor = tuple(np.argwhere(mask)[0])
    lnpi[anchor] = 0.0
    computed[anchor] = True

    queue = deque([anchor])

    while queue:
        current = queue.popleft()

        for dim in range(dims):
            minus = 1 + 2 * dim
            plus = 2 + 2 * dim

            # Move to the lower neighbor along this dimension.
            if current[dim] > 0:
                lower = list(current)
                lower[dim] -= 1
                lower = tuple(lower)

                if (
                    mask[lower]
                    and not computed[lower]
                    and p[lower][plus] > 0.0
                    and p[current][minus] > 0.0
                ):
                    lnpi[lower] = lnpi[current] - np.log(p[lower][plus] / p[current][minus])
                    computed[lower] = True
                    queue.append(lower)

            # Move to the upper neighbor along this dimension.
            if current[dim] < sizes[dim] - 1:
                upper = list(current)
                upper[dim] += 1
                upper = tuple(upper)

                if (
                    mask[upper]
                    and not computed[upper]
                    and p[current][plus] > 0.0
                    and p[upper][minus] > 0.0
                ):
                    lnpi[upper] = lnpi[current] + np.log(p[current][plus] / p[upper][minus])
                    computed[upper] = True
                    queue.append(upper)

    return log_normalize(lnpi)


def read_tmmc_results(
    limits,
    root_dir=".",
    statistics_file="tmmc/tmmc_statistics.txt",
    n_columns=13,
    max_total_particles=None,
):
    """
    Read `tmmc_statistics.txt` for every accepted grid point into a dense array.

    Grid points with `sum(particles) > max_total_particles` are skipped and are
    not counted as missing simulations.

    Returns
    -------
    tmmc_results : ndarray
        Dense result array with `NaN` entries where results are missing or skipped.
    missing : list[pathlib.Path]
        Expected result files for accepted grid points that were not found.
    skipped : list[pathlib.Path]
        Result paths excluded by the total-particle cutoff.
    """
    root_path = Path(root_dir)
    shape = tuple(len(limit) for limit in limits)
    tmmc_results = np.full((*shape, n_columns), np.nan)
    missing = []
    skipped = []

    for index in np.ndindex(shape):
        particles = [int(limits[dim][index[dim]]) for dim in range(len(limits))]
        simdir = root_path / "_".join(str(n) for n in particles)
        path = simdir / statistics_file

        if not particle_tuple_allowed(particles, max_total_particles=max_total_particles):
            skipped.append(path)
            continue

        if not path.exists():
            missing.append(path)
            continue

        values = np.genfromtxt(path)
        values = np.asarray(values, dtype=float).reshape(-1)

        if values.size < n_columns:
            raise ValueError(f"{path} has {values.size} columns; expected at least {n_columns}")

        tmmc_results[index] = values[:n_columns]

    print(f"Read {np.isfinite(tmmc_results[..., 0]).sum()} completed simulations")
    print(f"Missing {len(missing)} accepted simulations")

    if max_total_particles is not None:
        print(
            f"Skipped {len(skipped)} grid points with total particles "
            f"> {max_total_particles}"
        )

    return tmmc_results, missing, skipped


In [ ]:
tmmc_results, missing, skipped = read_tmmc_results(
    limits,
    root_dir=output_dir,
    max_total_particles=max_total_particles,
)

cmatrix = tmmc_results[..., 3:8]
pmatrix = normalize_cmatrix(cmatrix)
lnpi = optimize_pi(cmatrix)
pi = np.exp(lnpi)

print("pi normalization:", pi.sum())
print("non-zero pi states:", np.count_nonzero(pi))


## Fugacity reweighting

This section implements the reweighting equation defined above:

$$
\ln \tilde{\Pi}_{\mathbf{f}'}(\mathbf{N})
=
\ln \Pi_{\mathbf{f}}(\mathbf{N})
+
\sum_i N_i\ln\left(\frac{f_i'}{f_i}\right).
$$

The function `reweighted` accepts either a scalar fugacity, applied to every component, or a per-component vector such as `[f_h2o, f_co2]`. It returns normalized log probabilities at the target fugacity.


In [ ]:
def reweighted(tmmc_results, referenceFugacity, targetFugacity, limits):
    """
    Reweight the reference log distribution from `referenceFugacity` to `targetFugacity`.

    Parameters
    ----------
    tmmc_results : ndarray
        Dense TMMC result array returned by `read_tmmc_results`.
    referenceFugacity : float or sequence of float
        Reference fugacity or per-component reference fugacities.
    targetFugacity : float or sequence of float
        Target fugacity or per-component target fugacities.
    limits : list[ndarray]
        Particle-number values used for each component.

    Returns
    -------
    lnpi_reweighted : ndarray
        Normalized log particle-number distribution at `targetFugacity`.
    """
    dims = len(limits)
    n_transitions = 1 + 2 * dims

    cmatrix = tmmc_results[..., 3 : 3 + n_transitions]
    lnpi_reweighted = optimize_pi(cmatrix)

    referenceFugacity = as_fugacity_array(referenceFugacity, dims, "referenceFugacity")
    targetFugacity = as_fugacity_array(targetFugacity, dims, "targetFugacity")
    grids = macrostate_grids(limits)

    # Work in log space to avoid overflow for large N * log(f_target / f_reference).
    for dim, grid in enumerate(grids):
        lnpi_reweighted += grid * np.log(targetFugacity[dim] / referenceFugacity[dim])

    return log_normalize(lnpi_reweighted)


def prepare_target_fugacities(targetFugacities, dims):
    """
    Convert target fugacity inputs into an `(n_points, dims)` array.

    - A one-dimensional array longer than `dims` is interpreted as a scalar fugacity scan
      applied to every component.
    - A length-`dims` array is interpreted as one per-component target state.
    - A two-dimensional array must already have one column per component.
    """
    targetFugacities = np.asarray(targetFugacities, dtype=float)

    if targetFugacities.ndim == 0:
        targetFugacities = targetFugacities.reshape(1, 1)

    if targetFugacities.ndim == 1:
        if targetFugacities.size == dims:
            targetFugacities = targetFugacities.reshape(1, dims)
        else:
            targetFugacities = np.repeat(targetFugacities[:, None], dims, axis=1)

    if targetFugacities.ndim != 2 or targetFugacities.shape[1] != dims:
        raise ValueError(f"targetFugacities must have shape `(n_points, {dims})`")

    if np.any(targetFugacities <= 0.0):
        raise ValueError("targetFugacities must contain strictly positive fugacities")

    return targetFugacities


def mixture_isotherm(tmmc_results, referenceFugacity, targetFugacities, limits):
    """
    Compute mean loading for each component at one or more target fugacities.

    Returns
    -------
    mixture_isotherm : ndarray
        Array with shape `(n_target_fugacities, n_components)`.
    """
    dims = len(limits)
    targetFugacities = prepare_target_fugacities(targetFugacities, dims)
    referenceFugacity = as_fugacity_array(referenceFugacity, dims, "referenceFugacity")
    grids = macrostate_grids(limits)

    mixture_isotherm = np.zeros((targetFugacities.shape[0], dims), dtype=float)

    for i, targetFugacity in enumerate(targetFugacities):
        lnpi_reweighted = reweighted(
            tmmc_results=tmmc_results,
            referenceFugacity=referenceFugacity,
            targetFugacity=targetFugacity,
            limits=limits,
        )
        pi_reweighted = np.exp(lnpi_reweighted)

        for dim, grid in enumerate(grids):
            mixture_isotherm[i, dim] = np.nansum(pi_reweighted * grid)

    return mixture_isotherm


def loading_moments(pi, limits):
    """
    Compute mean loadings and the particle-number covariance matrix.

    The covariance matrix is useful because
    d<N_i>/d ln(f_j) = cov(N_i, N_j).
    """
    pi = np.asarray(pi, dtype=float)
    grids = macrostate_grids(limits)
    dims = len(grids)

    means = np.array([np.nansum(pi * grid) for grid in grids], dtype=float)
    covariance = np.zeros((dims, dims), dtype=float)

    for i in range(dims):
        for j in range(dims):
            covariance[i, j] = np.nansum(pi * grids[i] * grids[j]) - means[i] * means[j]

    return means, covariance


## Plot formatting helpers

The plotting functions below use consistent axis labels, color bars, grid alignment, and layout. The heatmaps are plotted on cell edges so each color cell is centered on the corresponding particle number.


In [ ]:
def set_plot_style():
    """Apply notebook-level Matplotlib formatting."""
    plt.rcParams.update(
        {
            "figure.dpi": 120,
            "savefig.dpi": 300,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.grid": False,
            "axes.labelsize": 11,
            "axes.titlesize": 12,
            "legend.frameon": False,
            "xtick.labelsize": 10,
            "ytick.labelsize": 10,
        }
    )


def grid_edges(values):
    """Convert grid-center values into pcolormesh edges."""
    values = np.asarray(values, dtype=float)

    if values.size == 1:
        return np.array([values[0] - 0.5, values[0] + 0.5])

    midpoints = 0.5 * (values[:-1] + values[1:])
    edges = np.empty(values.size + 1, dtype=float)
    edges[1:-1] = midpoints
    edges[0] = values[0] - (midpoints[0] - values[0])
    edges[-1] = values[-1] + (values[-1] - midpoints[-1])

    return edges


def format_grid_axis(ax, limits, component_labels):
    """Apply common labels and limits to a two-component particle-number heatmap."""
    x_edges = grid_edges(limits[0])
    y_edges = grid_edges(limits[1])

    ax.set_xlabel(fr"$N$ {component_labels[0]}")
    ax.set_ylabel(fr"$N$ {component_labels[1]}")
    ax.set_xlim(x_edges[0], x_edges[-1])
    ax.set_ylim(y_edges[0], y_edges[-1])


def plot_tmmc_summary(limits, tmmc_results, pi, component_labels=None):
    """Plot average energy, relative free energy, and particle-number probability."""
    if len(limits) != 2:
        raise ValueError("plot_tmmc_summary currently supports two-component grids")

    component_labels = component_labels or [f"component {i}" for i in range(len(limits))]

    free_energy = np.full_like(pi, np.nan, dtype=float)
    nonzero = pi > 0.0
    if np.any(nonzero):
        free_energy[nonzero] = -np.log(pi[nonzero])
        free_energy[nonzero] -= np.nanmin(free_energy[nonzero])

    x_edges = grid_edges(limits[0])
    y_edges = grid_edges(limits[1])

    fig, ax = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)

    pc_eng = ax[0].pcolormesh(x_edges, y_edges, tmmc_results[..., -1].T, shading="auto")
    pc_fe = ax[1].pcolormesh(x_edges, y_edges, free_energy.T, shading="auto")
    pc_pi = ax[2].pcolormesh(x_edges, y_edges, pi.T, shading="auto")

    ax[0].set_title("Average energy")
    ax[1].set_title(r"Relative free energy, $-\log(\Pi)$")
    ax[2].set_title(r"Particle-number distribution, $\Pi$")

    for axis in ax:
        format_grid_axis(axis, limits, component_labels)

    fig.colorbar(pc_eng, ax=ax[0], label="Average energy / K")
    fig.colorbar(pc_fe, ax=ax[1], label=r"$-\log(\Pi) - \min[-\log(\Pi)]$")
    fig.colorbar(pc_pi, ax=ax[2], label=r"$\Pi$")

    return fig, ax


def plot_reweighted_distribution(
    limits,
    tmmc_results,
    referenceFugacity,
    targetFugacity,
    component_labels=None,
):
    """Plot the distribution after fugacity reweighting."""
    lnpi_reweighted = reweighted(
        tmmc_results=tmmc_results,
        referenceFugacity=referenceFugacity,
        targetFugacity=targetFugacity,
        limits=limits,
    )
    pi_reweighted = np.exp(lnpi_reweighted)

    fig, ax = plot_tmmc_summary(
        limits=limits,
        tmmc_results=tmmc_results,
        pi=pi_reweighted,
        component_labels=component_labels,
    )

    targetFugacity = as_fugacity_array(targetFugacity, len(limits), "targetFugacity")
    target_label = ", ".join(f"{value:.3g}" for value in targetFugacity)
    fig.suptitle(f"Reweighted distribution at target fugacity [{target_label}]", y=1.05)

    return fig, ax, pi_reweighted


def plot_mixture_isotherm(fugacities, isotherm, component_labels=None):
    """Plot mean loading versus fugacity for each component."""
    fugacities = np.asarray(fugacities, dtype=float)
    isotherm = np.asarray(isotherm, dtype=float)

    if fugacities.ndim != 1:
        raise ValueError("plot_mixture_isotherm expects a one-dimensional fugacity grid")

    component_labels = component_labels or [f"component {i}" for i in range(isotherm.shape[1])]

    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)

    for dim, label in enumerate(component_labels):
        ax.plot(fugacities, isotherm[:, dim], marker="o", markersize=3, linewidth=1.5, label=label)

    ax.set_xscale("log")
    ax.set_xlabel("Fugacity / Pa")
    ax.set_ylabel("Mean loading / molecules per simulation cell")
    ax.set_title("Mixture adsorption isotherm")
    ax.grid(True, which="both", alpha=0.25)
    ax.legend(title="Component")

    return fig, ax


set_plot_style()


## Plot the reference distribution

The first plot shows the average energy stored in the last column of `tmmc_statistics.txt`. The second plot shows the relative dimensionless free-energy surface. The third plot shows the normalized particle-number distribution.

Use the free-energy plot to identify dominant macrostates and barriers. Use the probability plot to check whether the sampled distribution is localized, broad, or disconnected.


In [ ]:
fig, ax = plot_tmmc_summary(
    limits=limits,
    tmmc_results=tmmc_results,
    pi=pi,
    component_labels=component_labels,
)


## Reweighted distribution example

The example below reweights the reference distribution from `referenceFugacity` to `targetFugacity`. A scalar target fugacity is applied to both components. To set different component fugacities, use a vector, for example:

```python
targetFugacity = [9.7e3, 2.0e4]
```


In [ ]:
referenceFugacity = 9.7e4
targetFugacity = 9.7e3

lnpi_reweighted = reweighted(
    tmmc_results=tmmc_results,
    referenceFugacity=referenceFugacity,
    targetFugacity=targetFugacity,
    limits=limits,
)
pi_reweighted = np.exp(lnpi_reweighted)

print("reweighted pi normalization:", pi_reweighted.sum())

fig, ax, pi_reweighted = plot_reweighted_distribution(
    limits=limits,
    tmmc_results=tmmc_results,
    referenceFugacity=referenceFugacity,
    targetFugacity=targetFugacity,
    component_labels=component_labels,
)


## Mixture isotherm example

This cell scans a scalar fugacity from \(10^1\) to \(10^5\) Pa and applies the same scalar value to both components. For composition-dependent scans, pass an array with shape `(n_points, n_components)` to `mixture_isotherm`; each row is one target fugacity vector \(\mathbf{f}'^{(k)}\).

The returned `isotherm[k, i]` is

$$
\langle N_i\rangle_{\mathbf{f}'^{(k)}}
=
\sum_{\mathbf{N}}N_i\Pi(\mathbf{N};\mathbf{f}'^{(k)},V,T).
$$


In [ ]:
fugacities = np.logspace(1, 6, 100)

isotherm = mixture_isotherm(
    tmmc_results=tmmc_results,
    referenceFugacity=referenceFugacity,
    targetFugacities=fugacities,
    limits=limits,
)

fig, ax = plot_mixture_isotherm(
    fugacities=fugacities,
    isotherm=isotherm,
    component_labels=component_labels,
)


## Interpreting analysis problems

If many states in `pi` are zero, check whether the corresponding simulations exist and whether their transition counts are non-zero. The reconstruction requires connected paths through neighboring macrostates; isolated states cannot be assigned a probability from detailed-balance ratios.

If `pmatrix` contains rows of zeros, the corresponding `cmatrix` row had no counts. This usually indicates a missing or failed simulation, or a simulation that did not collect TMMC statistics.

If the free-energy plot has abrupt jumps, inspect the local transition counts. Very small counts make detailed-balance ratios noisy. Increasing `NumberOfCycles` or reducing `deltaN` can improve continuity, depending on the system.

If the energy plot has `NaN` regions inside the accepted grid, the notebook did not find the corresponding `tmmc/tmmc_statistics.txt` files. `NaN` regions outside the accepted grid are expected when `max_total_particles` is active.

If a high-fugacity reweighted distribution accumulates at the largest sampled value of one component, increase that component's `nmax` and rerun the missing high-loading part of the grid. If the distribution accumulates along the diagonal boundary created by `max_total_particles`, increase `max_total_particles`. A clipped boundary biases both \(\Pi\) and the isotherm.


## Notebook walkthrough

1. **Prepare `base/`:** Put a complete working default simulation in `base/`, including `simulation.json`, component definitions, the framework CIF, and force-field JSON files.
2. **Set the grid:** Edit `minmax_particles`, `component_ids`, `deltaN`, and `max_total_particles`. Confirm that the order of `component_ids` matches the order of particle ranges.
3. **Dry run `create`:** Run the dry-run cell to confirm the directory names and total number of simulations.
4. **Create directories:** Run the real creation cell with `dry_run=False`. Each folder receives a copied and edited `simulation.json`.
5. **Run simulations externally:** Execute the simulation engine in each generated folder.
6. **Read results:** Run `read_tmmc_results`. Missing accepted simulations are counted; states above `max_total_particles` are skipped separately.
7. **Reconstruct distribution:** Build `cmatrix`, normalize it into `pmatrix`, and compute `pi` with `optimize_pi`.
8. **Inspect plots:** Use the energy, free-energy, and probability plots to check convergence, missing regions, and dominant particle-number states.
9. **Reweight and compute isotherms:** Use `reweighted` for individual fugacity states and `mixture_isotherm` for fugacity scans.
